In [ ]:
from pathlib import Path
from PIL import Image
from pixel_arena.dataset_utils.celeb_a_mask_hq import get_prompt
from google.genai import types
from io import BytesIO
from clients import gemini_clients

original_image_path = Path(
    "eval-set/celeb/images-150/0700985d198843cbb080fbb525d44012.jpg"
)
color_palette_path = Path("label_palettes/seg_labels_celeb.png")
label_colors = None

MODEL = "gemini-3-pro-image-preview"
client = gemini_clients[0]

In [ ]:
def to_pil_image(image: types.Image) -> Image.Image:
    bytes = image.image_bytes
    return Image.open(BytesIO(bytes))

In [ ]:
original_file_name = original_image_path.stem


contents = [
    # first image is the original image
    Image.open(original_image_path).convert("RGB"),
    # second image is the color palette, as mentioned in the prompt
    Image.open(color_palette_path).convert("RGB"),
    get_prompt(label_colors),
]

thinking_config = (
    types.ThinkingConfig(include_thoughts=True)
    if MODEL == "gemini-3-pro-image-preview"
    else None
)
image_size = "1K" if MODEL == "gemini-3-pro-image-preview" else None

thinking_process = []
results = []


response = await client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=types.GenerateContentConfig(
        temperature=1.0,
        response_modalities=[
            "IMAGE",
            "TEXT",
        ],
        image_config=types.ImageConfig(
            aspect_ratio="1:1",
            image_size=image_size,
        ),
        top_p=0.95,
        thinking_config=thinking_config,
    ),
)

for part in response.parts:
    if part.thought:
        if part.text:
            thinking_process.append(part.text)
        elif image := part.as_image():
            thinking_process.append(image)
    if part.text is not None:
        results.append(part.text)
    elif image := part.as_image():
        results.append(image)
    else:
        pass

display(to_pil_image(thinking_process[1]))
display(to_pil_image(results[1]))

In [ ]:
print(thinking_process)

In [ ]:
print(results)

In [ ]:
import pickle

with open(
    "saved_result_binary/error_cases/response_of_error_case_partial_wrong_left_right.pkl",
    "wb",
) as f:
    pickle.dump(response, f)